# **SENTIMENT ANALYSIS**

## Import Necessary Libraries
 We imported standard data manipulation tools, machine learning metrics from sklearn, and Plotly Express (plotly.express) to generate interactive plots that handle hover actions natively.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## Load the Dataset and Initial Inspection
 We loaded the Twitter_Data.csv file into a Pandas DataFrame and reviewed its basic structure to ensure columns are parsed properly.

In [2]:
# Load dataset
df = pd.read_csv('Twitter_Data.csv')

# View data summary
print("Dataset Head:\n", df.head())
print("\nDataset Shape:", df.shape)

Dataset Head:
                                           clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0

Dataset Shape: (162980, 2)


## Data Cleansing and Handling Missing Values

We removed missing values from the dataset to prevent encoding errors during text transformation.


In [3]:
# Drop rows with missing text or categories
df = df.dropna().reset_index(drop=True)
print(f"Shape of the dataset after dropping missing values: {df.shape}")

Shape of the dataset after dropping missing values: (162969, 2)


## Feature Engineering — Word Count Extraction
What we did in this cell: We engineered a new text_length column by splitting the text into individual words, allowing us to display metadata in our hover charts.

In [4]:
# Calculate word counts for each clean text sample
df['text_length'] = df['clean_text'].apply(lambda x: len(str(x).split()))
print(df[['clean_text', 'text_length']].head())

                                          clean_text  text_length
0  when modi promised “minimum government maximum...           33
1  talk all the nonsense and continue all the dra...           13
2  what did just say vote for modi  welcome bjp t...           22
3  asking his supporters prefix chowkidar their n...           34
4  answer who among these the most powerful world...           14


## Sampling Data for Performance Optimization

Because rendering 160,000 text points dynamically with interactive html objects can slow down your browser, we extracted a random, well-balanced sample of 1,500 records specifically for the scatter visualizations

##Interactive Sentiment Category Distribution Chart (Hover Enabled)

 We created an interactive bar chart using Plotly where moving your cursor over a bar reveals the exact total count of that specific sentiment category.

In [5]:
# Sample 500 rows from each category to create a smooth interactive visual experience
sampled_df = df.groupby('category').apply(lambda x: x.sample(n=500, random_state=42)).reset_index(drop=True)
print(f"Sampled Dataset Shape for Visuals: {sampled_df.shape}")

Sampled Dataset Shape for Visuals: (1500, 3)


/tmp/ipykernel_4994/4157778555.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('category').apply(lambda x: x.sample(n=500, random_state=42)).reset_index(drop=True)


## Interactive Sentiment Category Distribution Chart (Hover Enabled)
 We created an interactive bar chart using Plotly where moving your cursor over a bar reveals the exact total count of that specific sentiment category.

In [6]:
# Map categories to readable string names for clear charts
df['sentiment_name'] = df['category'].map({-1.0: 'Negative', 0.0: 'Neutral', 1.0: 'Positive'})

# Create interactive bar chart
fig_bar = px.histogram(df, x='sentiment_name', color='sentiment_name',
                       title='Interactive Distribution of Sentiments (Hover to see Counts)',
                       labels={'sentiment_name': 'Sentiment Class'},
                       color_discrete_sequence=px.colors.qualitative.Pastel)

fig_bar.update_layout(showlegend=False)
fig_bar.show()

## Interactive Scatter Plot with Live Hover Details
We mapped out individual sample records onto a scatter plot where hovering over any single point instantly displays the actual text of the tweet and its exact word count.

In [7]:
# Map categories on sampled data
sampled_df['sentiment_name'] = sampled_df['category'].map({-1.0: 'Negative', 0.0: 'Neutral', 1.0: 'Positive'})

# Add a unique jitter/index tracker to spread points horizontally for easier hovering
sampled_df['tweet_index'] = sampled_df.index

# Build a scatter plot showing text content on cursor hover
fig_scatter = px.scatter(sampled_df, x='tweet_index', y='text_length', color='sentiment_name',
                         title='Interactive Tweet Analysis Plot (Hover over points to read the Tweet Text!)',
                         labels={'tweet_index': 'Tweet Sample Index', 'text_length': 'Word Count'},
                         hover_data={'clean_text': True, 'text_length': True, 'tweet_index': False},
                         color_discrete_sequence=px.colors.qualitative.Set2)

# Adjust plot markers for transparency and size
fig_scatter.update_traces(marker=dict(size=8, opacity=0.7))
fig_scatter.show()

## Split Data, Feature Vectorization, and Model Training
 We split the global dataset, converted the raw text tokens into TF-IDF numerical values, and trained our Logistic Regression model.

In [8]:
# Split features and labels
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['category'], test_size=0.2, random_state=42)

# Perform TF-IDF extraction
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Classifier
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Model Accuracy: 91.59%


##Interactive Confusion Matrix Heatmap (Hover Enabled)
 We calculated the model's accuracy matrix and wrapped it inside an interactive Plotly Heatmap so moving your cursor across blocks displays precise classifications.

In [9]:
# Build confusion matrix array
cm = confusion_matrix(y_test, y_pred)
labels = ['Negative', 'Neutral', 'Positive']

# Generate interactive Confusion Matrix Heatmap
fig_matrix = px.imshow(cm, x=labels, y=labels,
                       labels=dict(x="Predicted Label", y="True Label", color="Match Count"),
                       text_auto=True, # Display counts right inside the blocks
                       title="Interactive Confusion Matrix Heatmap (Hover to see Precision Mix)",
                       color_continuous_scale='Blues')

fig_matrix.show()